In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname  = "LIG",       # resname of your ligand in the topology
    topology_glob   = "*.pdb",
    trajectory_glob = "*.xtc",
    dt_ns           = 2.0,
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# Water bridge detection settings
WATER_RESNAMES   = None    # None = default (HOH, WAT, SOL, TIP3, TIP4, TIP)
MAX_BRIDGE_DIST  = 3.5     # Å — max O–O distance for both H-bond legs

OUTPUT_DIR = Path("./figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

## Step 1 — Run water bridge analysis for all replicas

In [ ]:
from mdatools.analysis.water_bridges import WaterBridgeAnalyzer
from mdatools.io.loaders import discover_replicas
from mdatools.universe import load_and_align

analyzer = WaterBridgeAnalyzer(
    cfg,
    water_resnames=WATER_RESNAMES,
    max_bridge_dist=MAX_BRIDGE_DIST,
)

results = {}
for rep in discover_replicas(REPLICA_ROOTS, cfg):
    u = load_and_align(rep["topology"], rep["trajectory"], cfg)
    results[rep["name"]] = analyzer.run(u, rep["name"])

for name, res in results.items():
    print(f"{name}: {len(res.events)} bridge events, {len(res.summary)} unique bridges")

## Step 2 — Bridge occupancy bar chart

In [ ]:
from mdatools.plotting.water_bridge_plots import plot_bridge_occupancy

for name, res in results.items():
    fig = plot_bridge_occupancy(
        res,
        top_n=10,
        save_path=OUTPUT_DIR / f"bridge_occupancy_{name}.png",
    )
    display(fig)

## Step 3 — Bridge formation timeline

In [ ]:
from mdatools.plotting.water_bridge_plots import plot_bridge_timeline

first_name, first_res = next(iter(results.items()))
fig = plot_bridge_timeline(
    first_res,
    top_n=5,
    save_path=OUTPUT_DIR / f"bridge_timeline_{first_name}.png",
)
fig

## Step 4 — Summary table

In [ ]:
for name, res in results.items():
    print(f"\n=== {name} ===")
    if res.summary.empty:
        print("  No water bridges detected.")
    else:
        display(res.summary.head(10))